# 🎓 Дообучение Джарвиса на ваших данных (Google Colab)

Ноутбук обучает **LoRA-адаптер** для модели **Qwen3 8B** на вашем датасете из приложения
(страница «Обучение» → «Скачать пакет для Colab») и превращает результат в файл **GGUF**,
который можно подключить к Ollama на своём компьютере.

**Что такое LoRA и QLoRA простыми словами.** Переобучать все 8 миллиардов параметров долго и
нужно 60+ ГБ видеопамяти. Вместо этого модель «замораживают» и учат маленькие добавочные
матрицы (адаптер, ~0,5% параметров). QLoRA — то же самое, но замороженная модель хранится в
4 битах, поэтому всё влезает в бесплатную видеокарту Colab (Tesla T4, 16 ГБ).

### Перед запуском
1. **Среда выполнения → Сменить среду выполнения → T4 GPU** (без видеокарты обучение невозможно).
2. Запускайте ячейки **по порядку** (Shift+Enter). Когда попросит — загрузите архив
   `jarvis_colab_package.zip` из приложения (или просто `train_sft.jsonl`).
3. Время: установка ~3 мин, обучение 200 примеров ~10–20 мин, экспорт GGUF ~15–25 мин.
4. Итог — файл `jarvis-qwen3.Q4_K_M.gguf` (~5 ГБ) на вашем Google Диске в папке `jarvis/`.

> Ваш датасет загружается в Google Colab — то есть **покидает компьютер**. Если в диалогах есть
> личное, просмотрите датасет в приложении перед экспортом. Локальная альтернатива —
> `training/train_local.py` (экспериментально, только Qwen3-4B на 8 ГБ).

## 1. Проверяем видеокарту

In [ ]:
# Должна быть Tesla T4 (или лучше). Если «command not found» / ошибка — включите GPU (см. выше)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Устанавливаем Unsloth
Unsloth ускоряет обучение в ~2 раза и экономит память. Установка занимает 2–4 минуты.

In [ ]:
%%capture
!pip install unsloth

## 3. Загружаем датасет
Выберите архив **`jarvis_colab_package.zip`** из приложения (в нём датасет и настройки)
или файл **`train_sft.jsonl`** (лежит в папке `data/dataset/` проекта).

In [ ]:
import json
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall(".")
        print(f"Распакован архив {name}:", ", ".join(zipfile.ZipFile(name).namelist()))
    else:
        print(f"Загружен файл {name}")

## 4. Настройки обучения
Если загружен пакет из приложения, значения подставятся из `config.json` автоматически
(приложение подбирает число эпох под размер датасета). Менять что-то вручную не обязательно.

In [ ]:
# --- значения по умолчанию ---
BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # Qwen3 8B, уже сжатая до 4 бит для QLoRA
DATASET_PATH = "train_sft.jsonl"
OUTPUT_NAME = "jarvis-qwen3"   # так будут называться файлы и модель в Ollama
MAX_SEQ_LENGTH = 2048          # длиннее — больше памяти; примеры длиннее будут пропущены
EPOCHS = 3                     # сколько раз модель увидит каждый пример
LEARNING_RATE = 2e-4           # скорость обучения (стандарт для LoRA)
LORA_R = 16                    # «ёмкость» адаптера: 8–32; больше — запоминает больше, но легче переобучиться

# --- значения из пакета приложения ---
if Path("config.json").exists():
    cfg = json.loads(Path("config.json").read_text(encoding="utf-8"))
    for key, value in cfg.items():
        if key.upper() in globals():
            globals()[key.upper()] = value
    print("Настройки из config.json:", cfg)

print(f"Модель: {BASE_MODEL} | эпох: {EPOCHS} | контекст: {MAX_SEQ_LENGTH} | LoRA r={LORA_R}")

## 5. Проверяем датасет
Функции ниже читают JSONL, отбрасывают битые строки и дубликаты, откладывают 10% примеров для проверки.

In [ ]:
import hashlib
import random

ROLES = {"system", "user", "assistant"}


def load_jsonl(path):
    """Читает датасет в формате {"messages": [...]}. Возвращает (диалоги, ошибки)."""
    rows, errors = [], []
    for number, line in enumerate(Path(path).read_text(encoding="utf-8-sig").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"строка {number}: это не JSON ({e.msg})")
            continue
        messages = row.get("messages") if isinstance(row, dict) else None
        if not isinstance(messages, list) or len(messages) < 2:
            errors.append(f"строка {number}: нет списка messages")
            continue
        if any(not isinstance(m, dict) or m.get("role") not in ROLES or not isinstance(m.get("content"), str)
               or not m["content"].strip() for m in messages):
            errors.append(f"строка {number}: неизвестная роль или пустое сообщение")
            continue
        if messages[-1]["role"] != "assistant" or messages[-2]["role"] != "user":
            errors.append(f"строка {number}: в конце должны быть вопрос пользователя и ответ ассистента")
            continue
        rows.append(messages)
    return rows, errors


def dedupe(rows):
    """Убирает повторы (одинаковые вопрос + ответ) — иначе модель их зазубрит."""
    seen, result = set(), []
    for messages in rows:
        key = hashlib.md5(json.dumps(messages[-2:], ensure_ascii=False).encode()).hexdigest()
        if key not in seen:
            seen.add(key)
            result.append(messages)
    return result


def split(rows, eval_share=0.1, seed=3407):
    """Откладывает часть примеров для проверки. Если примеров меньше 50 — проверочной части нет."""
    rows = rows[:]
    random.Random(seed).shuffle(rows)
    if len(rows) < 50:
        return rows, []
    n_eval = max(1, int(len(rows) * eval_share))
    return rows[n_eval:], rows[:n_eval]

In [ ]:
conversations, errors = load_jsonl(DATASET_PATH)
for e in errors[:10]:
    print("⚠", e)
before = len(conversations)
conversations = dedupe(conversations)
train_rows, eval_rows = split(conversations)

print(f"Примеров: {before}, после удаления дублей: {len(conversations)}, ошибок: {len(errors)}")
print(f"Обучение: {len(train_rows)}, проверка: {len(eval_rows)}")
if len(conversations) < 50:
    print("⚠ Примеров мало: модель скорее зазубрит их, чем научится. Для заметного эффекта нужно 200+.")
assert conversations, "Датасет пуст — проверьте, что загрузили train_sft.jsonl"
print("\nПример:", json.dumps(conversations[0][-2:], ensure_ascii=False, indent=1)[:800])

## 6. Загружаем модель
Скачивается ~6 ГБ, 2–5 минут.

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # сам выберет: на T4 и RTX 20xx — float16 (bfloat16 они не умеют)
    load_in_4bit=True,   # QLoRA: замороженная модель в 4 битах
)
print("bfloat16:", is_bfloat16_supported(), "| память GPU занята:",
      round(torch.cuda.memory_reserved() / 1024**3, 1), "ГБ")

## 7. Добавляем LoRA-адаптер
Обучаться будут только маленькие матрицы в слоях внимания и MLP.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,          # масштаб адаптера; обычно = r
    lora_dropout=0,             # 0 — быстрее всего в Unsloth
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # экономит память ценой ~20% скорости
    random_state=3407,
)
model.print_trainable_parameters()

## 8. Как модель отвечает ДО обучения
Сохраним ответы на несколько вопросов из датасета, чтобы потом сравнить.

In [ ]:
def ask(messages, max_new_tokens=350):
    """Ответ модели на диалог (без «размышлений», как в приложении по умолчанию)."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.8,
                            top_k=20, do_sample=True)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


test_rows = (eval_rows or train_rows)[:3]
FastLanguageModel.for_inference(model)
answers_before = [ask(msgs[:-1]) for msgs in test_rows]
for msgs, answer in zip(test_rows, answers_before):
    print("❓", msgs[-2]["content"][:200], "\n🤖 ДО:", answer[:500], "\n")
FastLanguageModel.for_training(model)

## 9. Готовим тексты для обучения
Каждый диалог превращается в текст по шаблону чата Qwen3 — ровно в том виде, в каком модель
видит его в Ollama. Слишком длинные примеры пропускаем.

In [ ]:
from datasets import Dataset


def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False,
                                         enable_thinking=False)


def build(rows):
    texts = [to_text(m) for m in rows]
    kept = [t for t in texts if len(tokenizer(t).input_ids) <= MAX_SEQ_LENGTH]
    if len(kept) < len(texts):
        print(f"Пропущено длинных примеров: {len(texts) - len(kept)} (длиннее {MAX_SEQ_LENGTH} токенов)")
    return Dataset.from_dict({"text": kept})


train_ds = build(train_rows)
eval_ds = build(eval_rows) if eval_rows else None
print(train_ds)
print(train_ds[0]["text"][:600])

## 10. Обучение
Следите за **loss** (ошибкой): она должна снижаться. Если на проверочных примерах (eval_loss)
ошибка начинает расти, а на обучающих продолжает падать — модель переобучается (зубрит);
тогда уменьшите EPOCHS.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # 2 × 4 = 8 примеров на шаг
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        eval_strategy="epoch" if eval_ds is not None else "no",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Учим только на ответах ассистента: вопросы и системный промпт модель не должна «заучивать»
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

stats = trainer.train()
print(f"Готово за {stats.metrics['train_runtime'] / 60:.1f} мин, итоговая loss: {stats.metrics['train_loss']:.3f}")
print("Пик памяти GPU:", round(torch.cuda.max_memory_reserved() / 1024**3, 1), "ГБ")

## 11. Как модель отвечает ПОСЛЕ обучения

In [ ]:
FastLanguageModel.for_inference(model)
for msgs, before_answer in zip(test_rows, answers_before):
    print("❓", msgs[-2]["content"][:200])
    print("🤖 ДО:   ", before_answer[:400])
    print("🎓 ПОСЛЕ:", ask(msgs[:-1])[:400])
    print("✅ ОБРАЗЕЦ:", msgs[-1]["content"][:400], "\n")

## 12. Сохраняем адаптер LoRA
Маленький файл (~150 МБ) — пригодится, если захотите продолжить обучение позже.

In [ ]:
model.save_pretrained(f"{OUTPUT_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_NAME}-lora")
!zip -qr {OUTPUT_NAME}-lora.zip {OUTPUT_NAME}-lora
print("Сохранено:", f"{OUTPUT_NAME}-lora.zip")

## 13. Экспорт в GGUF для Ollama
Адаптер «вклеивается» в модель, и результат сжимается в **Q4_K_M** — тот же формат, что у
`qwen3:8b` в Ollama, поэтому модель займёт столько же видеопамяти (~5,9 ГБ на вашей RTX 2060 Super).
Это самый долгий шаг: 15–25 минут.

In [ ]:
model.save_pretrained_gguf(f"{OUTPUT_NAME}-gguf", tokenizer, quantization_method="q4_k_m")

# Unsloth в разных версиях кладёт файл в разные места — ищем самый свежий *Q4_K_M*.gguf
candidates = sorted(Path(".").rglob("*.gguf"), key=lambda p: p.stat().st_mtime)
candidates = [p for p in candidates if "q4_k_m" in p.name.lower()] or candidates
gguf_path = candidates[-1]
final_path = Path(f"{OUTPUT_NAME}.Q4_K_M.gguf")
if gguf_path != final_path:
    gguf_path.rename(final_path)
print(f"GGUF: {final_path} ({final_path.stat().st_size / 1024**3:.1f} ГБ)")

## 14. Копируем результат на Google Диск
Скачивать 5 ГБ прямо из Colab ненадёжно, поэтому кладём файлы на Диск — оттуда скачайте
папку `jarvis/` на компьютер обычным способом.

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/jarvis")
dest.mkdir(parents=True, exist_ok=True)
for path in [final_path, Path(f"{OUTPUT_NAME}-lora.zip")]:
    shutil.copy(path, dest / path.name)
    print("На Диске:", dest / path.name)

## 15. (Необязательно) Лёгкий вариант: адаптер в GGUF (~100 МБ)
Вместо полной модели можно подключить к уже скачанной `qwen3:8b` только адаптер. Скачивать
намного меньше, **но** поддержку адаптеров для Qwen3 в Ollama я не гарантирую: если
подключение в приложении выдаст ошибку — используйте полный GGUF из шага 13.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q gguf
!python /content/llama.cpp/convert_lora_to_gguf.py {OUTPUT_NAME}-lora --base-model-id Qwen/Qwen3-8B --outtype f16 --outfile {OUTPUT_NAME}-lora.gguf
shutil.copy(f"{OUTPUT_NAME}-lora.gguf", dest / f"{OUTPUT_NAME}-lora.gguf")
print("На Диске:", dest / f"{OUTPUT_NAME}-lora.gguf")

## 16. Подключаем модель на компьютере
1. Скачайте `jarvis-qwen3.Q4_K_M.gguf` с Google Диска (папка `jarvis/`).
2. В приложении: **Обучение → Подключить модель** — укажите путь к файлу и нажмите
   «Подключить к Ollama». Приложение скопирует шаблон чата с `qwen3:8b`, чтобы у новой
   модели работали инструменты и «размышления».
3. Выберите **jarvis-qwen3** в списке моделей и сравните с `qwen3:8b` кнопкой «Повторить».

Вручную (без приложения): создайте файл `Modelfile` рядом с GGUF
```
FROM ./jarvis-qwen3.Q4_K_M.gguf
```
скопируйте в него строки `TEMPLATE` и `PARAMETER` из вывода `ollama show qwen3:8b --modelfile`
и выполните `ollama create jarvis-qwen3 -f Modelfile`.

**Если стало хуже:** уменьшите EPOCHS (модель переобучилась) или соберите больше разнообразных
примеров. Обучение — эксперимент: сравнивайте с исходной моделью на своих задачах.